# Réseau de neurones simple avec Keras

Dans ces travaux pratiques, nous allons construire avec Keras des réseaux de neurones qui reconnaissent les chiffres manuscrits de MNIST :

1. une régression logistique multinomiale, le réseau sans couche cachée, et l'accord entre sa sortie et sa perte ;
2. un perceptron multicouche régularisé, et les images qu'il classe mal ;
3. le suivi de l'entraînement avec TensorBoard, l'arrêt précoce et la sauvegarde du meilleur modèle ;
4. une recherche d'hyper-paramètres avec KerasTuner.

Il n'est pas nécessaire d'utiliser une carte graphique pour ce TP : chaque entraînement est rapide sur CPU, et seule la recherche d'hyper-paramètres, la dernière partie, prend quelques minutes. Vous pouvez donc changer d'environnement si celui qui vous a été attribué a une carte graphique (`Exécution > Modifier le type d'exécution`).

In [ ]:
!pip install -q keras-tuner
import keras
import keras_tuner
import matplotlib.pyplot as plt
import numpy as np
from keras import layers

# Mêmes tirages aléatoires à chaque exécution : des résultats reproductibles
keras.utils.set_random_seed(0)

## Récupération des données

Cette fois, nous récupérons le jeu de données par Keras.

In [ ]:
(X_train, y_train), (X_test, y_test) = keras.datasets.mnist.load_data()

## Regardons les données

Étudiez comment sont stockées les données :

- la forme et le type des quatre tableaux ;
- la valeur des premières étiquettes et l'intervalle des pixels ;
- quelques images, avec leur étiquette ;
- le nombre d'exemples de chaque classe : les classes sont-elles équilibrées ?

**Documentation utile**

- [`numpy.bincount`](https://numpy.org/doc/stable/reference/generated/numpy.bincount.html)
- [`matplotlib.axes.Axes.imshow`](https://matplotlib.org/stable/api/_as_gen/matplotlib.axes.Axes.imshow.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les attributs et les appels à utiliser :

- `X_train.shape`, `X_train.dtype`, `X_train.min()` et `X_train.max()`
- `np.bincount(y_train)` : le nombre d'occurrences de chaque entier, de 0 à 9
- `plt.subplots(1, 10)` renvoie une figure et 10 axes ; `ax.imshow(image, cmap="gray_r")` affiche une image 28 × 28 en niveaux de gris

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
print(X_train.shape, X_test.shape, y_train.shape, y_test.shape)
print(X_train.dtype, X_train.min(), X_train.max(), y_train[:10])
print(np.bincount(...))

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(...)
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
print(f"Format de X_train : {X_train.shape}, de X_test : {X_test.shape}")
print(f"Format de y_train : {y_train.shape}, de y_test : {y_test.shape}")
print(f"Type des images : {X_train.dtype}, de {X_train.min()} à {X_train.max()}")
print(f"Type des étiquettes : {y_train.dtype}, premières : {y_train[:10]}")
print(f"Exemples par classe : {np.bincount(y_train)}")

# 60 000 images d'entraînement et 10 000 de test, de 28 × 28 pixels ; des entiers
# non signés sur 8 bits (uint8), de 0 à 255 ; des étiquettes entières, de 0 à 9 ;
# de 5 421 à 6 742 exemples par classe : des classes à peu près équilibrées

In [ ]:
fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in enumerate(axes):
  ax.imshow(X_train[i], cmap="gray_r")
  ax.set_title(y_train[i])
  ax.axis("off")
plt.show()

## Transformation des données

Nous devons transformer ces données :

1. Aplatir les images : un tableau de forme `(_, 784)` plutôt que `(_, 28, 28)`, où `_` est le nombre d'exemples. Chaque image devient un vecteur de 784 pixels, l'entrée attendue par une couche `Dense`.
2. Ramener les pixels dans l'intervalle $[0, 1]$ plutôt que $[0, 255]$.
3. Encoder les classes en [*one-hot*](https://fr.wikipedia.org/wiki/Encodage_one-hot) : la première des deux pertes que nous verrons, `categorical_crossentropy`, attend pour chaque exemple 10 valeurs, toutes nulles sauf un 1 à l'indice de sa classe.

Effectuez les deux premières transformations sur `X_train` et `X_test`. Pour la troisième, gardez `y_train` et `y_test`, et rangez les classes encodées dans `Y_train` et `Y_test`.

**Documentation utile**

- [`keras.utils.to_categorical`](https://keras.io/api/utils/python_utils/#tocategorical-function)

In [ ]:
nb_classes = 10
dim_entree = 28 * 28

# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `X_train.reshape(-1, dim_entree)` : `-1` laisse NumPy calculer le nombre d'exemples
- `/ 255` divise chaque pixel et donne des flottants
- `keras.utils.to_categorical(y_train, num_classes=nb_classes)` renvoie un tableau de forme `(60000, 10)` ; l'étiquette 5 y devient `[0, 0, 0, 0, 0, 1, 0, 0, 0, 0]`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
X_train = X_train.reshape(...) / ...
X_test = ...
Y_train = keras.utils.to_categorical(..., num_classes=nb_classes)
Y_test = ...
```

### Solution

In [ ]:
# Des images « à plat » : l'entrée du réseau est un vecteur de input_dim pixels,
# dans [0, 1]. -1 laisse NumPy calculer le nombre d'exemples.
X_train = X_train.reshape(-1, dim_entree) / 255
X_test = X_test.reshape(-1, dim_entree) / 255

# Les classes en vecteurs one-hot, pour categorical_crossentropy
Y_train = keras.utils.to_categorical(y_train, num_classes=nb_classes)
Y_test = keras.utils.to_categorical(y_test, num_classes=nb_classes)
print(X_train.shape, X_test.shape, Y_train.shape, Y_test.shape)
print(f"Étiquette {y_train[0]} en one-hot : {Y_train[0]}")

## Régression logistique multinomiale

Créez avec `keras.Sequential` un modèle sans couche cachée, `modele` : une entrée `keras.Input` de `dim_entree` pixels, puis une couche `layers.Dense` de `nb_classes` neurones, avec l'activation softmax. Chaque neurone calcule une combinaison linéaire des pixels, et la softmax en fait 10 probabilités de somme 1 : c'est une régression logistique multinomiale.

Affichez son résumé avec `summary()` et expliquez son nombre de paramètres.

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.Sequential([...])` prend la liste des couches, dans l'ordre
- `keras.Input(shape=(dim_entree,))` : la forme d'**un** exemple, sans la taille du batch
- `layers.Dense(nb_classes, activation="softmax")`
- Résultat attendu : 7 850 paramètres

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele = keras.Sequential([keras.Input(shape=...), layers.Dense(..., activation=...)])
modele.summary()
```

### Solution

In [ ]:
modele = keras.Sequential(
  [keras.Input(shape=(dim_entree,)), layers.Dense(nb_classes, activation="softmax")]
)
modele.summary()

# Une couche Dense a un poids par entrée et par neurone, plus un biais par neurone :
# 784 × 10 + 10 = 7 850 paramètres. None est la taille du batch, libre.

## Apprentissage

Compilez le modèle avec `compile` : l'optimiseur `"sgd"` (descente de gradient stochastique, pas d'apprentissage de 0,01), la perte `categorical_crossentropy`, qui prend les cibles one-hot `Y_train`, et la métrique `"accuracy"`. Entraînez-le ensuite avec `fit`, avec :

- 128 exemples par batch ;
- 10 epochs ;
- 20 % des données d'entraînement en validation (`validation_split`).

Enfin, mesurez sa perte et son accuracy sur le jeu de test avec la fonction `evalue` ci-dessous. Gardez ces chiffres : ils servent de référence aux modèles suivants.

In [ ]:
def evalue(modele: keras.Model, one_hot: bool = False) -> None:
  """Affiche la perte et l'accuracy de `model` sur le jeu de test.

  one_hot : True pour un modèle entraîné sur les cibles one-hot (Y_train).
  """
  perte, exactitude = modele.evaluate(X_test, Y_test if one_hot else y_test, verbose=0)
  print(f"Test : perte {perte:.3f}, accuracy {exactitude:.3f}")

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])`
- `modele.fit(X_train, Y_train, batch_size=128, epochs=10, validation_split=0.2)` renvoie un objet `History`, dont l'attribut `history` donne la perte et les métriques de chaque epoch
- `evalue(modele, one_hot=True)` : le modèle apprend sur des cibles one-hot
- Résultat attendu : environ 89 % d'accuracy en validation comme en test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
modele.compile(optimizer=..., loss=..., metrics=[...])
historique = modele.fit(X_train, ..., batch_size=..., epochs=..., validation_split=...)
evalue(modele, one_hot=True)
```

### Solution

In [ ]:
# compile associe au modèle un optimiseur, une fonction de perte et des métriques
modele.compile(optimizer="sgd", loss="categorical_crossentropy", metrics=["accuracy"])

# fit entraîne le modèle compilé : une boucle d'entraînement complète
historique = modele.fit(
  X_train, Y_train, batch_size=128, epochs=10, validation_split=0.2
)
evalue(modele, one_hot=True)

## Accorder la sortie et la perte

La perte `sparse_categorical_crossentropy` calcule la même entropie croisée que `categorical_crossentropy`, mais à partir des étiquettes entières `y_train` : l'encodage one-hot devient inutile, et les modèles suivants s'en passent.

Elle attend par défaut des probabilités, la sortie d'une softmax. Une couche sans activation renvoie des *logits*, les scores bruts des classes : la perte `keras.losses.SparseCategoricalCrossentropy(from_logits=True)` applique alors elle-même la softmax, en un calcul plus stable.

Entraînez sur `y_train`, avec l'optimiseur `"adam"` et les réglages de la partie précédente, trois régressions logistiques :

1. `modele_adam`, le modèle de la régression logistique, avec la perte `sparse_categorical_crossentropy` ;
2. `modele_logits`, la même couche sans activation, avec `from_logits=True` ;
3. `modele_faux`, la même couche sans activation, avec la perte par défaut : la mauvaise paire.

Comparez leur perte et leur accuracy en test : quelle version apprend mal ? Pourquoi Keras ne signale-t-il aucune erreur ?

**Documentation utile**

- [`keras.losses.SparseCategoricalCrossentropy`](https://keras.io/api/losses/probabilistic_losses/#sparsecategoricalcrossentropy-class)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- un modèle neuf par version : le même `keras.Sequential` que `modele`, avec `activation="softmax"` ou sans activation (`activation=None`, la valeur par défaut)
- `loss="sparse_categorical_crossentropy"` attend des probabilités ; `loss=keras.losses.SparseCategoricalCrossentropy(from_logits=True)`, des logits
- `fit(..., verbose=0)` n'affiche pas les epochs ; `evalue(...)` sans `one_hot`, les cibles étant les étiquettes entières
- Une fonction `entraine(activation, perte)` qui crée, compile, entraîne et évalue un modèle évite d'écrire trois fois le même code
- Résultat attendu : environ 92 % d'accuracy en test pour `modele_adam`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def entraine(activation: str | None, perte: str | keras.losses.Loss) -> keras.Model:
  modele = keras.Sequential(
    [keras.Input(shape=(dim_entree,)), layers.Dense(nb_classes, activation=...)]
  )
  modele.compile(optimizer="adam", loss=..., metrics=["accuracy"])
  modele.fit(
    X_train, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0
  )
  evalue(modele)
  return modele


modele_adam = entraine("softmax", "sparse_categorical_crossentropy")
modele_logits = entraine(None, ...)
modele_faux = entraine(..., ...)
```

### Solution

In [ ]:
def entraine(activation: str | None, perte: str | keras.losses.Loss) -> keras.Model:
  """Entraîne et évalue une régression logistique, avec Adam, sur y_train."""
  modele = keras.Sequential(
    [keras.Input(shape=(dim_entree,)), layers.Dense(nb_classes, activation=activation)]
  )
  modele.compile(optimizer="adam", loss=perte, metrics=["accuracy"])
  modele.fit(
    X_train, y_train, batch_size=128, epochs=10, validation_split=0.2, verbose=0
  )
  evalue(modele)
  return modele


# Bonne paire : une sortie softmax, la perte par défaut, sur des probabilités
modele_adam = entraine("softmax", "sparse_categorical_crossentropy")
# Bonne paire : une sortie sans activation, des logits, et from_logits=True
sur_logits = keras.losses.SparseCategoricalCrossentropy(from_logits=True)
modele_logits = entraine(None, sur_logits)
# Mauvaise paire : des logits, que la perte par défaut lit comme des probabilités
modele_faux = entraine(None, "sparse_categorical_crossentropy")

# Les deux bonnes paires donnent les mêmes chiffres : environ 92 % en test, 3 points
# de plus que la régression logistique entraînée avec SGD, à architecture égale. La
# mauvaise paire n'atteint qu'environ 40 %, avec une perte bloquée vers 2,3. Keras ne
# voit pas d'erreur : des logits ont la forme et le type de probabilités. La perte
# par défaut borne chaque valeur entre 1e-7 et 1 - 1e-7 avant d'en prendre le
# logarithme : un logit négatif devient 1e-7, un logit au-dessus de 1 presque 1, et
# le gradient s'annule pour ces valeurs. L'autre mauvaise paire, une sortie softmax
# avec from_logits=True, Keras la repère sous TensorFlow : il prévient (UserWarning)
# et reprend les logits d'avant la softmax.

## Perceptron multicouche régularisé

Écrivez une fonction `construit_modele_profond` qui renvoie un perceptron multicouche neuf, non compilé :

- 4 couches cachées de 20 neurones, avec l'activation ReLU ;
- sur leurs poids, l'initialisation de He (`kernel_initializer`) et une pénalité L2 de coefficient $10^{-4}$ (`kernel_regularizer`) : une option de la couche, qui ajoute à la perte la somme des carrés de ses poids pour freiner le sur-apprentissage ;
- la même couche de sortie softmax que la régression logistique.

Créez-en un, `modele_profond`, affichez son résumé, compilez-le avec l'optimiseur `"adam"`, la perte `sparse_categorical_crossentropy` et l'accuracy, puis entraînez-le avec les mêmes réglages que la régression logistique. Combien de points d'accuracy en test gagne-t-il sur `modele_adam`, la régression logistique entraînée avec le même optimiseur ? Pour combien de paramètres de plus ? Comparez aussi l'accuracy d'entraînement et celle de validation de la dernière epoch : ce réseau sur-apprend-il ?

Les parties suivantes réutilisent `construit_modele_profond`.

In [ ]:
def construit_modele_profond() -> keras.Model: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `kernel_regularizer=keras.regularizers.L2(1e-4)` ajoute à la perte $10^{-4} \sum w^2$, sur les poids de la couche ; les biais ne sont pas pénalisés
- `kernel_initializer="he_normal"` : des poids initiaux tirés d'une loi normale de variance $2 / n_{\text{entrée}}$, adaptée à ReLU, qui met à zéro environ la moitié des activations
- Les quatre couches cachées ont les mêmes options : un dictionnaire `options_cachees`, passé avec `**options_cachees`, évite de les répéter
- `accuracy` et `val_accuracy`, à la dernière ligne qu'affiche `fit` : l'accuracy d'entraînement, moyenne sur l'epoch, et celle de validation
- Résultat attendu : 17 170 paramètres, environ 95 % d'accuracy en test

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_modele_profond() -> keras.Model:
  options_cachees = dict(activation=..., kernel_regularizer=..., kernel_initializer=...)
  return keras.Sequential(
    [
      keras.Input(shape=(dim_entree,)),
      layers.Dense(20, **options_cachees),
      ...,
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )


modele_profond = construit_modele_profond()
modele_profond.summary()
modele_profond.compile(optimizer=..., loss=..., metrics=["accuracy"])
modele_profond.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evalue(modele_profond)
```

### Solution

In [ ]:
def construit_modele_profond() -> keras.Model:
  """Un perceptron multicouche neuf : 4 couches cachées régularisées de 20 neurones."""
  options_cachees = dict(
    activation="relu",
    kernel_regularizer=keras.regularizers.L2(1e-4),
    kernel_initializer="he_normal",
  )
  return keras.Sequential(
    [
      keras.Input(shape=(dim_entree,)),
      layers.Dense(20, **options_cachees),
      layers.Dense(20, **options_cachees),
      layers.Dense(20, **options_cachees),
      layers.Dense(20, **options_cachees),
      layers.Dense(nb_classes, activation="softmax"),
    ],
    name="modele_profond",
  )


modele_profond = construit_modele_profond()
modele_profond.summary()
modele_profond.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
modele_profond.fit(X_train, y_train, batch_size=128, epochs=10, validation_split=0.2)
evalue(modele_profond)

# Environ 2,5 points de plus que modele_adam (95 % contre 92,4 %), pour 17 170
# paramètres au lieu de 7 850 : le gain de l'architecture seule, à optimiseur égal.
# Contre la régression logistique entraînée avec SGD, les 5 points d'écart mêlent
# architecture et optimiseur. À la dernière epoch, l'accuracy d'entraînement dépasse
# celle de validation d'un à deux points, et val_loss remonte depuis quelques epochs :
# un début de sur-apprentissage, que la pénalité L2 n'empêche pas à elle seule.
# L'arrêt précoce, plus loin, arrête l'entraînement à temps.

## Images mal classées

`evalue` résume le modèle en deux chiffres sur le jeu de test ; `predict` renvoie sa sortie pour chaque exemple, ici les 10 probabilités de chaque image de test. Avec `predict` et `np.argmax`, calculez la classe que `modele_profond` prédit pour chaque image de test, comptez les images mal classées et vérifiez que leur nombre s'accorde avec le résultat de `evalue`. Affichez ensuite 10 de ces images, avec leur étiquette et la prédiction. Ces erreurs vous semblent-elles excusables ?

**Documentation utile**

- [`keras.Model.predict`](https://keras.io/api/models/model_training_apis/#predict-method)
- [`numpy.argmax`](https://numpy.org/doc/stable/reference/generated/numpy.argmax.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `modele_profond.predict(X_test, verbose=0)` renvoie un tableau de forme `(10000, 10)` : une ligne de probabilités par image de test
- `np.argmax(probas, axis=1)` : pour chaque ligne, l'indice de la plus grande probabilité, la classe prédite
- `np.flatnonzero(predictions != y_test)` : les indices des images mal classées
- `X_test[i].reshape(28, 28)` remet en 28 × 28 pixels une image aplatie, pour `imshow`
- Résultat attendu : environ 500 images mal classées sur 10 000, les 5 % d'erreurs que donne `evalue`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
probas = modele_profond.predict(X_test, verbose=0)
predictions = np.argmax(..., axis=...)
erreurs = np.flatnonzero(...)
print(f"{len(erreurs)} images mal classées sur {len(y_test)}")

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in zip(erreurs[:10], axes):
  ax.imshow(..., cmap="gray_r")
  ax.set_title(f"étiquette {...}\nprédit {...}")
  ax.axis("off")
plt.show()
```

### Solution

In [ ]:
# predict renvoie la sortie du modèle pour chaque image : 10 probabilités par ligne
probas = modele_profond.predict(X_test, verbose=0)
# La classe prédite : l'indice de la plus grande probabilité de chaque ligne
predictions = np.argmax(probas, axis=1)
erreurs = np.flatnonzero(predictions != y_test)
print(f"{len(erreurs)} images mal classées sur {len(y_test)}")
print(f"Part bien classée : {1 - len(erreurs) / len(y_test):.3f}, comme dans evalue")

fig, axes = plt.subplots(1, 10, figsize=(14, 2))
for i, ax in zip(erreurs[:10], axes):
  # Les images ont été aplaties : reshape les remet en 28 × 28 pixels
  ax.imshow(X_test[i].reshape(28, 28), cmap="gray_r")
  ax.set_title(f"étiquette {y_test[i]}\nprédit {predictions[i]}")
  ax.axis("off")
plt.show()

# Environ 500 erreurs sur 10 000 : les 5 % que donne evalue, recalculés à la main.
# Certaines sont excusables, des chiffres mal formés qu'un humain hésiterait à lire
# (un 5 refermé comme un 6, un 4 aux traits ouverts) ; d'autres non, un 3 ou un 6
# bien lisibles. Le perceptron voit les 784 pixels à plat, sans la forme du tracé :
# un chiffre un peu décalé ou penché change beaucoup de ses entrées. Les réseaux
# convolutifs, conçus pour les images, évitent une bonne part de ces erreurs.

## Adam contre SGD sous TensorBoard

TensorBoard affiche les courbes d'apprentissage qu'un callback `keras.callbacks.TensorBoard`, passé à `fit`, écrit dans un dossier : un sous-dossier par entraînement. Lancez la cellule suivante avant l'exercice : le tableau de bord se remplit au fil des epochs (bouton de rechargement en haut à droite).

Entraînez ensuite deux perceptrons multicouches neufs (`construit_modele_profond()`) avec les réglages de la partie « Perceptron multicouche régularisé », l'un avec l'optimiseur `"adam"`, l'autre avec `"sgd"`. Passez à chaque `fit` un callback TensorBoard qui écrit dans `logs/adam` pour le premier, dans `logs/sgd` pour le second.

Comparez les courbes `epoch_loss` et `epoch_accuracy` des deux entraînements : lequel converge le plus vite ? Pourquoi ?

**Documentation utile**

- [`keras.callbacks.TensorBoard`](https://keras.io/api/callbacks/tensorboard/)

In [ ]:
%load_ext tensorboard
%tensorboard --logdir logs

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.callbacks.TensorBoard(log_dir="logs/adam")`, passé par `fit(..., callbacks=[...])`
- Un modèle neuf par optimiseur, `construit_modele_profond()` puis `compile` : sinon le second entraînement repart des poids du premier
- Dans TensorBoard, l'onglet *Scalars* superpose les courbes des deux dossiers ; `train` et `validation` séparent les courbes d'entraînement et de validation
- Relancer un entraînement ajoute ses courbes au même dossier : supprimez `logs` ou changez de nom de dossier pour repartir d'un tableau de bord propre

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
for optimiseur in ["adam", "sgd"]:
  modele = construit_modele_profond()
  modele.compile(
    optimizer=..., loss="sparse_categorical_crossentropy", metrics=["accuracy"]
  )
  tensorboard = keras.callbacks.TensorBoard(log_dir=...)
  modele.fit(X_train, y_train, ..., callbacks=[...])
```

### Solution

In [ ]:
for optimiseur in ["adam", "sgd"]:
  modele = construit_modele_profond()
  modele.compile(
    optimizer=optimiseur, loss="sparse_categorical_crossentropy", metrics=["accuracy"]
  )
  tensorboard = keras.callbacks.TensorBoard(log_dir=f"logs/{optimiseur}")
  modele.fit(
    X_train,
    y_train,
    batch_size=128,
    epochs=10,
    validation_split=0.2,
    callbacks=[tensorboard],
  )

# Adam converge bien plus vite : il dépasse 90 % d'accuracy en validation dès la
# première epoch, SGD seulement après plusieurs. SGD multiplie le gradient de chaque
# poids par le même pas d'apprentissage (0,01) ; Adam l'adapte à chaque poids, d'après
# la taille de ses gradients récents, et garde une inertie d'un pas à l'autre (la
# moyenne des gradients).

## Arrêt précoce et sauvegarde du meilleur modèle

Plutôt que de fixer le nombre d'epochs à l'avance, on arrête l'entraînement quand la perte de validation ne baisse plus. Deux callbacks s'en chargent, mais ne gardent pas la même chose.

Entraînez un perceptron multicouche neuf avec Adam, sur 50 epochs au plus, avec deux callbacks :

- `keras.callbacks.EarlyStopping` : arrêt après 3 epochs sans baisse de `val_loss`, **sans** `restore_best_weights` ;
- `keras.callbacks.ModelCheckpoint` : sauvegarde du modèle dans `meilleur.keras`, chaque fois que `val_loss` atteint un nouveau minimum.

Tracez ensuite `loss` et `val_loss`, d'après l'historique que renvoie `fit`, et marquez l'epoch du modèle sauvegardé. Enfin, rechargez ce modèle avec `keras.models.load_model`, et comparez son évaluation sur le test à celle du modèle en mémoire : de quelle epoch viennent les poids de chacun ? Que changerait `restore_best_weights=True` ?

**Documentation utile**

- [`matplotlib.pyplot.axvline`](https://matplotlib.org/stable/api/_as_gen/matplotlib.pyplot.axvline.html)

In [ ]:
# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `keras.callbacks.EarlyStopping(monitor="val_loss", patience=3)` : sans `restore_best_weights`, le modèle garde les poids de la dernière epoch
- `keras.callbacks.ModelCheckpoint("meilleur.keras", monitor="val_loss", save_best_only=True)`
- `fit(..., epochs=50, callbacks=[...])` renvoie un `History` : `historique.history["val_loss"]` donne une valeur par epoch, et sa longueur le nombre d'epochs effectuées
- `np.argmin(...)` renvoie l'indice du minimum, compté à partir de 0, quand Keras numérote les epochs à partir de 1 ; `plt.axvline(x)` trace une droite verticale en `x`
- `keras.models.load_model("meilleur.keras")` renvoie un modèle compilé, prêt pour `evalue`

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
# Graine remise avant de tirer les poids : les cellules précédentes ne comptent plus
keras.utils.set_random_seed(0)
modele = construit_modele_profond()
modele.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
rappels = [
  keras.callbacks.EarlyStopping(...),
  keras.callbacks.ModelCheckpoint(...),
]
historique = modele.fit(X_train, y_train, ..., callbacks=rappels)
nb_epochs = len(historique.history["loss"])
epoch_sauvegardee = int(np.argmin(...)) + 1

epochs = range(1, nb_epochs + 1)
plt.plot(epochs, historique.history["loss"], label="loss")
plt.plot(epochs, ..., label="val_loss")
plt.axvline(..., color="gray", linestyle="--", label="modèle sauvegardé")
plt.xlabel("epoch")
plt.legend()
plt.show()

evalue(modele)
evalue(keras.models.load_model(...))
```

### Solution

In [ ]:
# Graine remise avant de tirer les poids : les cellules précédentes ne comptent plus
keras.utils.set_random_seed(0)
modele = construit_modele_profond()
modele.compile(
  optimizer="adam", loss="sparse_categorical_crossentropy", metrics=["accuracy"]
)
rappels = [
  # Arrêt après 3 epochs sans baisse de val_loss : le modèle en mémoire garde les
  # poids de la dernière epoch
  keras.callbacks.EarlyStopping(monitor="val_loss", patience=3),
  # Sauvegarde du modèle à chaque nouveau minimum de val_loss
  keras.callbacks.ModelCheckpoint(
    "meilleur.keras", monitor="val_loss", save_best_only=True
  ),
]
historique = modele.fit(
  X_train,
  y_train,
  batch_size=128,
  epochs=50,
  validation_split=0.2,
  callbacks=rappels,
)
nb_epochs = len(historique.history["loss"])
# Le minimum de val_loss : l'epoch que ModelCheckpoint a sauvegardée en dernier
epoch_sauvegardee = int(np.argmin(historique.history["val_loss"])) + 1
print(f"Epochs effectuées : {nb_epochs}, epoch sauvegardée : {epoch_sauvegardee}")

epochs = range(1, nb_epochs + 1)
plt.plot(epochs, historique.history["loss"], label="loss")
plt.plot(epochs, historique.history["val_loss"], label="val_loss")
plt.axvline(epoch_sauvegardee, color="gray", linestyle="--", label="modèle sauvegardé")
plt.xlabel("epoch")
plt.legend()
plt.show()

print("Modèle en mémoire :")
evalue(modele)
print("Modèle sauvegardé :")
modele_sauvegarde = keras.models.load_model("meilleur.keras")
evalue(modele_sauvegarde)

# Le modèle en mémoire a les poids de la dernière epoch, 3 epochs après le minimum
# de val_loss ; meilleur.keras, ceux du minimum. Leurs chiffres de test sont proches
# ici, val_loss remontant à peine en 3 epochs, mais rien ne garantit qu'ils le soient.
# EarlyStopping décide quand s'arrêter, ModelCheckpoint garde sur disque le meilleur
# modèle vu ; restore_best_weights=True remettrait aussi en mémoire les poids du
# minimum, ceux de meilleur.keras. Sur la courbe, loss baisse encore quand val_loss
# ne baisse plus : le début du sur-apprentissage.

## Recherche d'hyper-paramètres

Combien de neurones dans la couche cachée, quel pas d'apprentissage ? KerasTuner essaie des combinaisons d'hyper-paramètres et garde la meilleure, notée sur le jeu de validation.

Écrivez une fonction `construit_modele(hp)` qui renvoie un perceptron à une couche cachée ReLU, compilé avec Adam, la perte `sparse_categorical_crossentropy` et l'accuracy. Elle lit ses hyper-paramètres dans `hp` :

- `unites`, le nombre de neurones cachés : de 32 à 512, par pas de 32 ;
- `taux_apprentissage`, le pas d'apprentissage d'Adam : de $10^{-4}$ à $10^{-2}$, en échelle logarithmique.

Lancez ensuite la recherche avec `keras_tuner.BayesianOptimization` : contrairement à `RandomSearch`, qui tire chaque essai au hasard, il choisit le suivant d'après les résultats des précédents. Objectif `val_accuracy`, 5 essais, chacun entraîné 3 fois pour une moyenne moins bruitée, 10 epochs par entraînement, des batchs de 1 500 exemples et 20 % de validation. Affichez enfin le résumé des essais, puis le meilleur modèle, avec sa perte et son accuracy en test. Comparez-les à celles de `modele_profond`, puis comparez les essais entre eux : lequel des deux hyper-paramètres compte le plus, et pourquoi ?

C'est la partie la plus longue : quelques minutes sur CPU.

**Documentation utile**

- [`keras_tuner.HyperParameters`](https://keras.io/keras_tuner/api/hyperparameters/)
- [`keras_tuner.BayesianOptimization`](https://keras.io/keras_tuner/api/tuners/bayesian/#bayesianoptimization-class)

In [ ]:
def construit_modele(hp: keras_tuner.HyperParameters) -> keras.Model: ...


# Votre code ici

### Conseils supplémentaires

Les appels et les paramètres à utiliser :

- `hp.Int("unites", min_value=32, max_value=512, step=32)` et `hp.Float("taux_apprentissage", min_value=1e-4, max_value=1e-2, sampling="log")` renvoient la valeur de l'essai en cours
- `keras_tuner.BayesianOptimization(construit_modele, objective="val_accuracy", max_trials=5, executions_per_trial=3, seed=0, directory="tuner", project_name="mnist", overwrite=True)` : `overwrite=True` repart de zéro au lieu de reprendre la recherche déjà enregistrée dans `tuner/mnist`
- `tuner.search(...)` prend les arguments de `fit`
- `tuner.results_summary()` affiche les essais, du meilleur au moins bon ; `tuner.get_best_models()[0]` renvoie le meilleur modèle, déjà entraîné
- Pour l'expliquer, comptez les mises à jour des poids de chaque entraînement : les 48 000 exemples d'entraînement divisés par la taille des batchs, fois le nombre d'epochs
- Résultat attendu : de 94 % à 98 % d'accuracy en test pour le meilleur modèle, selon les essais tirés

Un squelette à copier dans la cellule de l'exercice, en remplaçant chaque `...` :

```python
def construit_modele(hp: keras_tuner.HyperParameters) -> keras.Model:
  unites = hp.Int("unites", ...)
  taux_apprentissage = hp.Float("taux_apprentissage", ...)
  modele = keras.Sequential(
    [
      keras.Input(shape=(dim_entree,)),
      layers.Dense(..., activation="relu"),
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )
  modele.compile(
    optimizer=keras.optimizers.Adam(...),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return modele


tuner = keras_tuner.BayesianOptimization(construit_modele, ...)
tuner.search(X_train, y_train, ...)
tuner.results_summary()
meilleur_modele = tuner.get_best_models()[0]
meilleur_modele.summary()
evalue(meilleur_modele)
```

### Solution

In [ ]:
def construit_modele(hp: keras_tuner.HyperParameters) -> keras.Model:
  # Chaque appel de hp lit la valeur que le tuner a choisie pour l'essai en cours
  unites = hp.Int("unites", min_value=32, max_value=512, step=32)
  taux_apprentissage = hp.Float(
    "taux_apprentissage", min_value=1e-4, max_value=1e-2, sampling="log"
  )
  modele = keras.Sequential(
    [
      keras.Input(shape=(dim_entree,)),
      layers.Dense(unites, activation="relu"),
      layers.Dense(nb_classes, activation="softmax"),
    ]
  )
  modele.compile(
    optimizer=keras.optimizers.Adam(taux_apprentissage),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
  )
  return modele


tuner = keras_tuner.BayesianOptimization(
  construit_modele,
  objective="val_accuracy",
  max_trials=5,
  executions_per_trial=3,
  seed=0,
  directory="tuner",
  project_name="mnist",
  overwrite=True,
)
tuner.search_space_summary()

In [ ]:
tuner.search(X_train, y_train, epochs=10, batch_size=1500, validation_split=0.2)

In [ ]:
tuner.results_summary()
meilleur_modele = tuner.get_best_models()[0]
meilleur_modele.summary()
evalue(meilleur_modele)

# Le pas d'apprentissage compte plus que le nombre de neurones. Avec des batchs de
# 1 500 exemples, 10 epochs ne font que 320 mises à jour des poids, contre 3 750 pour
# modele_profond (batchs de 128) : un petit pas n'a pas le temps de converger, quelle
# que soit la taille du modèle. Dans une exécution, les essais à moins de 1,2e-3
# restent entre 95,5 % et 97 % en validation, ceux au-dessus de 5e-3 dépassent
# 97,5 %, et le meilleur modèle (384 neurones, pas de 9e-3) fait 98 % en test, contre
# 95 % pour modele_profond, avec 18 fois plus de paramètres. Quand les essais tirés
# n'ont que de petits pas, il fait moins bien que modele_profond (94 % dans une autre
# exécution). La recherche classe les configurations pour ce budget d'entraînement,
# pas pour ce qu'elles donneraient entraînées plus longtemps.